# Agentic RL / RLVR：从 verifier 到多轮工具轨迹

本 Notebook 直接调用仓库中的真实函数，目标是观察：工具调用如何解析、执行与验证，strict/shaped reward 有何区别，action/observation mask 为什么不同。

建议先阅读 `docs/foundation_model_interview/02_AGENTIC_RLVR_FROM_ZERO_TO_MASTERY.md`。这里的随机/手工样例只验证代码逻辑，不代表模型能力。

In [ ]:
from pathlib import Path
import inspect
import json
import sys

repo = Path.cwd().resolve()
if not (repo / 'trainer').exists():
    repo = repo.parent
assert (repo / 'trainer' / 'train_agent.py').exists(), '请从仓库根目录或 learning_notebooks 目录运行'
sys.path.insert(0, str(repo))
print('repo =', repo)

## 1. 先看数据：最后一条 assistant 答案不会进入 rollout prompt

`AgentRLDataset` 返回 `messages[:-1]`、工具 schema 和 GT。GT 只交给 verifier，不直接告诉模型。

In [ ]:
fixture = repo / 'tests' / 'fixtures' / 'agent_rl_smoke.jsonl'
sample = json.loads(fixture.read_text(encoding='utf-8').splitlines()[0])
print('roles:', [m['role'] for m in sample['conversations']])
print('user:', sample['conversations'][-2]['content'])
print('held-out answer:', sample['conversations'][-1]['content'])
print('gt:', sample['gt'])

## 2. Tool call：可解析、参数合法、执行成功是三个不同条件

In [ ]:
from trainer.train_agent import (
    TOOLS, calculate_rewards, execute_tool, parse_tool_calls,
)

valid_call = '<tool_call>{"name":"calculate_math","arguments":{"expression":"2+2"}}</tool_call>'
bad_json = '<tool_call>{name: calculate_math}</tool_call>'
missing_argument = '<tool_call>{"name":"calculate_math","arguments":{}}</tool_call>'

for text in [valid_call, bad_json, missing_argument]:
    print(text, '=>', parse_tool_calls(text))
print('execution result:', execute_tool('calculate_math', {'expression': '2+2'}))

## 3. Verifiable reward：正确答案不能被格式分替代

下面比较合法轨迹、数字子串攻击和未闭合轨迹。`strict` 只输出任务成功的 `+1/-1`。

In [ ]:
math_tools = [t for t in TOOLS if t['function']['name'] == 'calculate_math']

def reward_case(final_answer, turns, gt=('4',), unfinished=False, mode='strict'):
    out = calculate_rewards(
        prompts=['What is 2+2?'],
        completions=[final_answer],
        gt_batch=[list(gt)],
        tools_batch=[math_tools],
        num_gen=1,
        device='cpu',
        turn_outputs_batch=[turns],
        unfinished_batch=[unfinished],
        require_tool_call_for_success=True,
        reward_mode=mode,
        return_details=True,
    )
    return {
        'reward': out.rewards.item(),
        'task_success': bool(out.task_success.item()),
        'answer_accuracy': out.answer_accuracy.item(),
        'format_valid': out.format_valid.item(),
        'tool_valid': out.tool_call_valid.item(),
        'execution_success': out.tool_execution_success.item(),
    }

cases = {
    'valid': reward_case('Final answer: 4', [valid_call, 'Final answer: 4']),
    'substring_attack': reward_case('Final answer: 14', [valid_call, 'Final answer: 14']),
    'unfinished': reward_case('Final answer: 4', [valid_call, 'Final answer: 4'], unfinished=True),
}
print(json.dumps(cases, ensure_ascii=False, indent=2))
assert cases['valid']['task_success']
assert not cases['substring_attack']['task_success']
assert not cases['unfinished']['task_success']

## 4. Strict 与 shaped 不可直接比较数值

shaped 用于课程学习，strict 用于公平 RLVR 主对照。即使 shaped reward 较高，只要最终成功条件不成立，`task_success` 仍应为 false。

In [ ]:
strict_wrong = reward_case('Final answer: 14', [valid_call, 'Final answer: 14'], mode='strict')
shaped_wrong = reward_case('Final answer: 14', [valid_call, 'Final answer: 14'], mode='shaped')
print('strict wrong:', strict_wrong)
print('shaped wrong:', shaped_wrong)
assert strict_wrong['task_success'] == shaped_wrong['task_success'] == False

## 5. Action mask：observation 能被看到，但不能当成模型 action

In [ ]:
symbolic_tokens = [
    ('system/user prompt', 0),
    ('assistant <tool_call>', 1),
    ('tool observation', 0),
    ('assistant final answer', 1),
    ('EOS', 1),
    ('PAD', 0),
]
for token_source, action_mask in symbolic_tokens:
    print(f'{token_source:28s} action_mask={action_mask}')
assert [mask for _, mask in symbolic_tokens] == [0, 1, 0, 1, 1, 0]

## 6. 读取真实函数，而不是 Notebook 副本

重点观察 `response_mask` 和 `response_old_logps` 在 action/observation 两处分配的值。

In [ ]:
from trainer.train_agent import rollout_single
source = inspect.getsource(rollout_single)
for line in source.splitlines():
    if any(key in line for key in ['response_mask', 'response_old_logps', 'messages.append', 'parse_tool_calls']):
        print(line)

## 7. 下一步实际运行

1. 运行 `python -m unittest tests.test_agent_rewards -v`。
2. 按主教程第 14 节运行 CPU smoke。
3. 在 SFT checkpoint 上打开 `--debug_mode --debug_interval 1`。
4. 人工核对至少 20～50 条 trajectory 后再启动正式训练。
5. 进阶：实现最终答案与 tool result 的一致性 verifier，并加入 observation 反事实测试。